#### common (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:= retry_fn' retries fn =
    :~ ^ 루프 retry =
        := is_error, result =
            ?= fn () ?|
            | Ok x => !-, x
            | Error x => !+, x
        ? not is_error || retry >= retries
        ?> result
        |:
            trace Debug
                \ () => "common.retry_fn\' / 루프"
                \ () =>
                    :< sm'_operators
                    { is_error retry = sm'::obj_to_string retry ++# "/" ++# sm'::obj_to_string retries; result }
            루프 (retry + 1)
    루프 1

:= upcast \* t u. (x : t) : u =
    $'!x :> `u '

:= downcast \* t u. (x : t) : u =
    $'!x :?> `u '

:! random = $'System.Random'

:= random () : random =
    $'`random ' ()

:= random_next (min : i32) (max : i32) (random : random) : i32 =
    $'!random.Next (!min, !max)'

:== disposable_switch t =
    {
        Gleam : $'Nil'
        Fsharp : $'System.IDisposable'
        Python : $'object'
        Rust : $'Rc<dyn Fn()>'
        TypeScript : $'() => void'
    }
:! disposable t = $'backend_switch `(disposable_switch t)'

:= dispose (disposable : disposable _) : () =
    backend_switch {
        Gleam = \ () => $'// common.dispose' : ()
        Lua = \ () => $'-- common.dispose' : ()
        Fsharp = \ () => disposable |> $'_.Dispose()' : ()
        Python = \ () => $'!disposable.__exit__(None, None, None)' : ()
        Rust = \ () => $'(!disposable)()' : ()
        TypeScript = \ () => $'(!disposable)()' : ()
    }

:= yield \* t. (x : t) : () =
    $'yield !x '

:= return \* t. (x : t) : () =
    $'return !x '

:= return' \* t. (x : t) : t =
    $'return !x '

:= return_in \* t. (x : t) : t =
    $'!x in return !x '

:= retry_fn \* t. retries (fn : () -> t) : option t =
    :~ ^ 루프 retry =
        try
            \ () =>
                ? retry < retries
                ?> fn () |> Some
                |: None
            \ ex =>
                trace Warning
                    \ () => "common.retry_fn"
                    \ () => { retry ex }
                None
        |> \|
            | Some x => x
            | None => 루프 (retry + 1)
    루프 0

#### retry_fn (test)

In [1]:
///- --test --symbol
///> cuda
///> rust
///> typescript

:= retry_fn_test = mut 0i32
\ () =>
    retry_fn_test <- *retry_fn_test + 1
    *retry_fn_test
|> retry_fn 3i32
|> _assert_eq' (Some 1i32)

.py (cuda)
__assert_eq' { expected = Some(1) }
.rs (rust)
__assert_eq' { expected = Some(1) }
.ts (typescript)
__assert_eq' { expected = Some(1) }

0

In [2]:
///- --test --symbol
///> rust
///> typescript
///> cuda

:= retry_fn_test = mut 0i32
\ () =>
    ? *retry_fn_test >= 2
    ?> *retry_fn_test
    |:
        retry_fn_test <- *retry_fn_test + 1
        failwith "test"
|> retry_fn 3i32
|> _assert_eq' (Some 2i32)

.rs (rust)
17:30:34 1w common.retry_fn / { retry = 0; ex = "test" }
17:30:34 2w common.retry_fn / { retry = 1; ex = "test" }
__assert_eq' { expected = Some(2) }
.ts (typescript)
17:30:37 1w common.retry_fn / { retry = 0; ex = Error: test }
17:30:37 2w common.retry_fn / { retry = 1; ex = Error: test }
__assert_eq' { expected = Some(2) }
.py (cuda)
17:30:41 1w common.retry_fn / { retry = 0; ex = test }
17:30:41 2w common.retry_fn / { retry = 1; ex = test }
__assert_eq' { expected = Some(2) }

0

In [ ]:
:= lua_math_random (low : i32) (high : i32) : i32 = $'math.random(!low, !high)' : i32

:= native_rng_state () : () =
    global "static SPIRAL_NATIVE_RNG: std::sync::atomic::AtomicU64 = std::sync::atomic::AtomicU64::new(0);"

:= native_rng_load () : u64 =
    native_rng_state ()
    $'SPIRAL_NATIVE_RNG.load(std::sync::atomic::Ordering::Relaxed)' : u64

:= native_rng_store (x : u64) : () = $'SPIRAL_NATIVE_RNG.store(!x, std::sync::atomic::Ordering::Relaxed)' : ()

:= xorshift (x : u64) : u64 =
    := x = x ^^^ (x <<< 13i32)
    := x = x ^^^ (x >>> 7i32)
    x ^^^ (x <<< 17i32)

:= native_rng_seed () : u64 =
    := a, _ = guid::rust_random_pair ()
    a ||| 1u64

:= native_rng_next () : u64 =
    := current = native_rng_load ()
    := next = xorshift (? current = 0u64 ?> native_rng_seed () |: current)
    native_rng_store next
    next

:= random_in_span (x : u64) (lo : i64) (hi : i64) : i64 =
    := span = hi - lo
    ? span <= 0i64 ?> lo |: lo + convert::i64 (x % convert::u64 span)

:= typescript_random_u53 () : u64 = $'BigInt(Math.floor(Math.random() * 9007199254740992))' : u64

:= native_random_between (lo : i64) (hi : i64) : i64 = random_in_span (native_rng_next ()) lo hi

:= process_counter_add (name : string) (n : u64) : u64 =
    run_target \|
        | Rust => \ () =>
            global """
                fn spiral_process_counter_add(name: &str, n: u64) -> u64 {
                    use std::collections::HashMap;
                    use std::sync::Mutex;
                    static COUNTERS: Mutex<Option<HashMap<String, u64>>> = Mutex::new(None);
                    let mut guard = COUNTERS.lock().unwrap();
                    let total = guard.get_or_insert_with(HashMap::new).entry(name.to_string()).or_insert(0);
                    *total += n;
                    *total
                }
                """
            $'spiral_process_counter_add(&*!name, !n)' : u64
        | _ => \ () => failwith "process_counter_add: Rust only; on the BEAM use an otp_actor counter"

:= process_counter_value (name : string) : u64 = process_counter_add name 0u64

#### process_counters_add_up_per_name (test)

In [3]:
///- --test --symbol
///> rust

:= ~a, b = "spiral-counter-test-a", "spiral-counter-test-b"
process_counter_add a 2u64 |> ignore
process_counter_add a 3u64 |> ignore
process_counter_add b 1u64 |> ignore
(process_counter_value a, process_counter_value b, process_counter_value "spiral-counter-test-none")
|> _assert_eq (5u64, 1u64, 0u64)

__assert_eq { expected = (5, 1, 0) }

0

In [ ]:
:= random' \* t. (min : t) (max : t) : t =
    run_target \|
        | Zig => \ () => random_in_span (guid::zig_random_u64 ()) (convert::i64 min) (convert::i64 max) |> convert::numeric_conv
        | Rust => \ () => native_random_between (convert::i64 min) (convert::i64 max) |> convert::convert
        | Cuda _ => \ () =>
            global "import random"
            $'random.randrange(!min, !max)' : t
        | TypeScript => \ () => random_in_span (typescript_random_u53 ()) (convert::i64 min) (convert::i64 max) |> convert::numeric_conv
        | Lua => \ () =>
            := low, high = convert::i32 min, convert::i32 max - 1i32
            convert::convert (? high < low ?> low |: lua_math_random low high)
        | _ => \ () =>
            random () |> random_next (convert::i32 min) (convert::i32 max) |> convert::convert

#### random_draws_stay_in_their_half_open_range (test)

In [4]:
///- --test --symbol
///> lua
///> rust

:~ ^ outside (draws : i32) : i32 =
    ? draws = 0i32 ?> 0i32
    |:
        := x = random' 3i32 7i32
        (? x < 3i32 || x >= 7i32 ?> 1i32 |: 0i32) + outside (draws - 1i32)
(outside 200i32, random' 5i32 5i32, random' 5i32 6i32) |> _assert_eq (0i32, 5i32, 5i32)

.lua (lua)
__assert_eq { expected = (0, 5, 5) }
.rs (rust)
__assert_eq { expected = (0, 5, 5) }

0

In [ ]:
:= new_disposable (fn : () -> ()) : disposable _ =
    run_target \|
        | Rust => \ () =>
            :< rust_operators
            !\\(fn, $'"$0.clone()"') : disposable _
        | TypeScript => \ () =>
            := fn = ~~ fn
            $'!fn ' : disposable _
        | Fsharp _ => \ () =>
            := fn = ~~ fn
            $'{ new System.IDisposable with member _.Dispose () = !fn () }'
        | Cuda _ => \ () =>
            $'class Disposable:'
            $'    def __init__(self, fn):'
            $'        self.fn = fn'
            $'    def __exit__(self, exc_type, exc_value, traceback):'
            $'        self.fn()'
            $'        return False'
            $'Disposable(!fn)'
        | _ => \ () => null ()

#### new_disposable (test)

In [5]:
///- --test --symbol
///> cuda
///> rust
///> typescript

:= new_disposable_test = mut 0i32
new_disposable \ () => new_disposable_test <- *new_disposable_test + 1
|> \ x => x : disposable ()
|> dispose
*new_disposable_test |> _assert_eq 1

.py (cuda)
__assert_eq { expected = 1 }
.rs (rust)
__assert_eq { expected = 1 }
.ts (typescript)
__assert_eq { expected = 1 }

0

In [6]:
///- --test --symbol
///> fsharp

:= new_disposable_test = mut 0i32
\ () =>
    new_disposable \ () => new_disposable_test <- *new_disposable_test + 1
    |> \ x => x : disposable ()
    |> use
    |> ignore
    |> return
|> async::new_task
|> async::await_task
|> async::run_synchronously
*new_disposable_test |> _assert_eq 1

__assert_eq { expected = 1 }

In [7]:
///- --test --symbol
///> fsharp

:= new_disposable_test = mut 0i32
\ () =>
    new_disposable \ () => new_disposable_test <- *new_disposable_test + 1
    |> \ x => x : disposable ()
    |> ignore
    |> return
|> async::new_task
|> async::await_task
|> async::run_synchronously
*new_disposable_test |> _assert_eq 0

__assert_eq { expected = 0 }

In [8]:
///> fsharp

:= main () =
    init_trace_state None
    := new_disposable x : _ () = new_disposable x
    $'let new_disposable x = !new_disposable x' : ()
    := retry_fn (r : i32) (x : () -> _) : optionm'::option' () = retry_fn r x |> optionm'::box
    $'let retry_fn x = !retry_fn x' : ()
    := memoize (fn : () -> ()) : () -> () = memoize fn
    $'let memoize x = !memoize x' : ()